In [1]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime

from data.utils import get_project_root, safe


ROOT_DIR = get_project_root()
BASE_PATH = os.path.join(ROOT_DIR, "data", "data_sirene")
SILVER_PATH = os.path.join(BASE_PATH, "formatted_data")
BASE_FAKE_PATH = os.path.join(BASE_PATH, "fake_data")
execution_date = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = os.path.join(ROOT_DIR, "data", "fake_data", "rib", execution_date)
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [2]:
import random
from faker import Faker

fake = Faker("fr_FR")

BANKS = [
    {"name": "BNP Paribas", "code_banque": "30004", "bic": "BNPAFRPP", "logo": "bnp.png"},
    {"name": "Société Générale", "code_banque": "30003", "bic": "SOGEFRPP", "logo": "sg.png"},
    {"name": "Crédit Agricole", "code_banque": "30006", "bic": "AGRIFRPP", "logo": "ca.png"},
    {"name": "La Banque Postale", "code_banque": "20041", "bic": "PSSTFRPP", "logo": "lbp.png"}
]

def compute_rib_key(bank, branch, account):
    def convert(c):
        if c.isdigit():
            return c
        return str(ord(c.upper()) - 55)

    account_num = "".join(convert(c) for c in account)
    rib_number = f"{bank}{branch}{account_num}"
    key = 97 - (int(rib_number) % 97)
    return f"{key:02d}"

def compute_iban(bank, branch, account, rib_key):
    def convert(c):
        if c.isdigit():
            return c
        return str(ord(c.upper()) - 55)

    account_num = "".join(convert(c) for c in account)

    rib = f"{bank}{branch}{account_num}{rib_key}"
    temp = rib + "1527"
    iban_key = 98 - (int(temp) % 97)
    return f"FR{iban_key:02d}{rib}"

def get_rib_holder(company):
    denom = company.get("denominationUniteLegale")

    if denom and denom.strip():
        return denom.strip(), "entreprise"

    nom = company.get("nomUniteLegale")
    prenom = company.get("prenom1UniteLegale") or company.get("prenomUsuelUniteLegale")

    if nom and prenom:
        return f"{prenom.strip()} {nom.strip()}", "personne"

    if nom:
        return nom.strip(), "personne"

    return "Titulaire Inconnu", "inconnu"

def get_address(company):
    return company.get("adresseEtablissement") or (
        f"{company.get('numeroVoieEtablissement','')}"+
        f"{company.get('typeVoieEtablissement','')}"+
        f"{company.get('libelleVoieEtablissement','')}, "+
        f"{company.get('codePostalEtablissement','')}"+
        f"{company.get('libelleCommuneEtablissement','')}"
    )

def generate_rib(company):
    bank = random.choice(BANKS)

    code_banque = bank["code_banque"]
    code_guichet = str(random.randint(10000, 99999))
    account = "".join(random.choices("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ", k=11))

    rib_key = compute_rib_key(code_banque, code_guichet, account)
    iban = compute_iban(code_banque, code_guichet, account, rib_key)

    holder, type_holder = get_rib_holder(company)

    return {
        "titulaire": holder,
        "type": type_holder,
        "adresse": get_address(company),

        "banque": bank["name"],
        "logo_banque": bank["logo"],
        "bic": bank["bic"],

        "code_banque": code_banque,
        "code_guichet": code_guichet,
        "numero_compte": account,
        "cle_rib": rib_key,
        "iban": iban,

        "agence": fake.city()
    }

In [3]:
!pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

def rib_to_pdf(rib, filename):
    doc = SimpleDocTemplate(filename)
    styles = getSampleStyleSheet()

    elements = []

    elements.append(Paragraph("<b>RELEVÉ D'IDENTITÉ BANCAIRE</b>", styles["Title"]))
    elements.append(Spacer(1, 12))

    elements.append(Paragraph(f"<b>Titulaire :</b> {rib['titulaire']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Adresse :</b> {rib['adresse']}", styles["Normal"]))
    elements.append(Spacer(1, 10))

    elements.append(Paragraph(f"<b>Banque :</b> {rib['banque']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Agence :</b> {rib['agence']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>BIC :</b> {rib['bic']}", styles["Normal"]))
    elements.append(Spacer(1, 10))

    elements.append(Paragraph(f"<b>Code Banque :</b> {rib['code_banque']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Code Guichet :</b> {rib['code_guichet']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Numéro de compte :</b> {rib['numero_compte']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Clé RIB :</b> {rib['cle_rib']}", styles["Normal"]))
    elements.append(Spacer(1, 10))

    elements.append(Paragraph(f"<b>IBAN :</b> {rib['iban']}", styles["Normal"]))

    doc.build(elements)


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
parquet_files = list(Path(SILVER_PATH).glob("*.parquet"))

if not parquet_files:
    print(f"Erreur : Aucun fichier Parquet trouvé dans : {SILVER_PATH}")
    exit(1)

input_file = max(parquet_files, key=lambda f: f.stat().st_mtime)
print(f"Fichier sélectionné : {input_file.name}")

try:
    df_silver = pd.read_parquet(input_file)
except Exception as e:
    print(f"Erreur lors de la lecture du Parquet : {e}")
    exit(1)

os.makedirs(OUTPUT_DIR, exist_ok=True)

limit = min(10, len(df_silver))
print(f"Génération de {limit} fichiers Kbis...")



Fichier sélectionné : sirene_full_20260317_164601.parquet
Génération de 10 fichiers Kbis...


In [6]:
from pathlib import Path

for index, row in df_silver.head(limit).iterrows():
    company = row.to_dict()

    if 'metadata' in company and isinstance(company['metadata'], dict):
        company.update(company['metadata'])

    siret = safe(company.get('siret', fake.numerify('#########')))
    rib = generate_rib(company)

    pdf_path = Path(OUTPUT_DIR) / f"rib_{siret}_{index:03d}.pdf"
    print(pdf_path)

    rib_to_pdf(rib, str(pdf_path))


C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\rib\20260317_172817\rib_21890197300048_002.pdf
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\rib\20260317_172817\rib_21890260900039_017.pdf
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\rib\20260317_172817\rib_21890103100011_030.pdf
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\rib\20260317_172817\rib_21890107200023_031.pdf
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\rib\20260317_172817\rib_21890294800031_032.pdf
